### Imports

In [ ]:
!pip install -q -U faiss-gpu-cu12

In [ ]:
import os
import json
import time
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from sentence_transformers import SentenceTransformer
import faiss

### Hyperparameter Configuration

In [ ]:
RAG_BUILD_CONFIG = {
    "train_csv": "train.csv",
    "test_csv": "test.csv",
    "wiki_api": "https://en.wikipedia.org/w/api.php",
    "titles_per_prompt": 2,
    "request_delay_sec": 0.1,
    "max_fetch_workers": 8,
    "chunk_words": 150,
    "chunk_overlap_words": 30,
    "embedding_model_name": "BAAI/bge-large-en-v1.5",
    "query_instruction": "Represent this sentence for searching relevant passages: ",
    "reranker_model_name": "BAAI/bge-reranker-base",
    "output_dir": "rag_wiki_bundle",
    "bundle_zip_name": "rag_wiki_bundle"
}

os.makedirs(RAG_BUILD_CONFIG["output_dir"], exist_ok=True)

### Loading Train + Test Prompts

In [ ]:
train_df = pd.read_csv(RAG_BUILD_CONFIG["train_csv"])
prompt_frames = [train_df[["prompt"]]]

try:
    test_df = pd.read_csv(RAG_BUILD_CONFIG["test_csv"])
    prompt_frames.append(test_df[["prompt"]])
    print(f"Loaded test.csv: {len(test_df)} rows")
except FileNotFoundError:
    print("test.csv not found -- proceeding with train prompts only")

all_prompts_df = pd.concat(prompt_frames, ignore_index=True)
raw_unique_prompts = all_prompts_df["prompt"].drop_duplicates().tolist()
print(f"Unique raw prompts (train + test): {len(raw_unique_prompts)}")

Loaded test.csv: 500 rows
Unique raw prompts (train + test): 1983


### Query Cleaning

In [ ]:
PREFIX_PHRASES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Determine the correct option:",
    "Identify the correct statement:",
    "Choose the correct answer:"
]

SUFFIX_PHRASES = [
    "among the listed options.",
    "based on the given context.",
    "from the following choices.",
    "carefully."
]

def clean_query(prompt):
    q = str(prompt).strip()
    for p in PREFIX_PHRASES:
        if q.startswith(p):
            q = q[len(p):].strip()
            break
    for s in SUFFIX_PHRASES:
        if q.endswith(s):
            q = q[:-len(s)].strip()
            break
    return q

In [ ]:
cleaned_to_raw = {}
for p in raw_unique_prompts:
    c = clean_query(p)
    cleaned_to_raw.setdefault(c, p)

unique_queries = list(cleaned_to_raw.keys())
print(f"Unique cleaned queries: {len(unique_queries)}")
print(f"Collapsed {len(raw_unique_prompts) - len(unique_queries)} near-duplicate prompts (same topic, different wrapper)")
print()
print("Example:")
print(" raw    :", raw_unique_prompts[0])
print(" cleaned:", clean_query(raw_unique_prompts[0]))

Unique cleaned queries: 424
Collapsed 1559 near-duplicate prompts (same topic, different wrapper)

Example:
 raw    : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
 cleaned: What is Martin Heidegger's view on the relationship between time and human existence?


### Wikipedia Session

In [ ]:
def build_wiki_session(pool_size=20):
    session = requests.Session()
    session.headers.update({
        "User-Agent": "SmartMCQSolverRAG/1.0 (Kaggle DLGenAI competition project)"
    })
    retry = Retry(
        total=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=pool_size, pool_maxsize=pool_size)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

wiki_session = build_wiki_session(pool_size=20)

### Wikipedia Search & Extract Helpers

In [ ]:
def wiki_search_titles(query, limit=2):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": limit,
        "format": "json",
    }
    try:
        r = wiki_session.get(RAG_BUILD_CONFIG["wiki_api"], params=params, timeout=10)
        r.raise_for_status()
        hits = r.json().get("query", {}).get("search", [])
        return [h["title"] for h in hits]
    except Exception:
        return None

def wiki_get_extract_single(title):
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "redirects": 1,
        "titles": title,
        "format": "json",
    }
    try:
        r = wiki_session.get(RAG_BUILD_CONFIG["wiki_api"], params=params, timeout=20)
        r.raise_for_status()
        pages = r.json().get("query", {}).get("pages", {})
        for page in pages.values():
            return title, (page.get("extract", "") or "")
    except Exception:
        pass
    return title, ""

### Resolving Cleaned Queries to Wikipedia Page Titles

In [ ]:
resolved_titles = set()
query_to_titles = {}
failed_queries = []

for query in tqdm(unique_queries, desc="Searching Wikipedia"):
    titles = wiki_search_titles(query, limit=RAG_BUILD_CONFIG["titles_per_prompt"])
    if titles is None:
        failed_queries.append(query)
        continue
    query_to_titles[query] = titles
    resolved_titles.update(titles)
    time.sleep(RAG_BUILD_CONFIG["request_delay_sec"])

resolved_titles = sorted(resolved_titles)
print(f"Unique Wikipedia pages resolved: {len(resolved_titles)}")
print(f"Failed searches: {len(failed_queries)} / {len(unique_queries)}")
if failed_queries:
    print("Sample failed queries (first 5):")
    for q in failed_queries[:5]:
        print(" -", q[:80])

Searching Wikipedia:   0%|          | 0/424 [00:00<?, ?it/s]

Unique Wikipedia pages resolved: 487
Failed searches: 0 / 424


### Fetching Article Extracts

In [ ]:
title_to_text = {}

with ThreadPoolExecutor(max_workers=RAG_BUILD_CONFIG["max_fetch_workers"]) as executor:
    futures = {executor.submit(wiki_get_extract_single, t): t for t in resolved_titles}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching extracts (parallel)"):
        title, text = future.result()
        if text.strip():
            title_to_text[title] = text

print(f"Articles with usable text: {len(title_to_text)} / {len(resolved_titles)}")

Fetching extracts (parallel):   0%|          | 0/487 [00:00<?, ?it/s]

Articles with usable text: 486 / 487


### Chunking Articles into Passages

In [ ]:
def chunk_text(text, title, chunk_words, overlap_words):
    words = text.split()
    chunks = []
    step = max(chunk_words - overlap_words, 1)
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_words]
        if len(piece) < 20:
            continue
        chunks.append({"source_title": title, "text": " ".join(piece)})
        if start + chunk_words >= len(words):
            break
    return chunks

passages = []
for title, text in tqdm(title_to_text.items(), desc="Chunking"):
    passages.extend(chunk_text(
        text, title,
        RAG_BUILD_CONFIG["chunk_words"],
        RAG_BUILD_CONFIG["chunk_overlap_words"]
    ))

passages_df = pd.DataFrame(passages)
passages_df.insert(0, "passage_id", range(len(passages_df)))
print(f"Total passages: {len(passages_df)}")
passages_df.head()

Chunking:   0%|          | 0/486 [00:00<?, ?it/s]

Total passages: 25577


,passage_id,source_title,text
0,0,2018 in science,A number of significant scientific events occu...
1,1,2018 in science,method to create stronger and more resilient n...
2,2,2018 in science,sizes and regular spacings. This could imply t...
3,3,2018 in science,"study published in the journal Cell, Universit..."
4,4,2018 in science,thereby potentially reducing drug side-effects...


### Embedding Passages

In [ ]:
embedder = SentenceTransformer(RAG_BUILD_CONFIG["embedding_model_name"])

passage_embeddings = embedder.encode(
    passages_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)
passage_embeddings = np.asarray(passage_embeddings, dtype="float32")
print(passage_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/800 [00:00<?, ?it/s]

(25577, 1024)


### Building FAISS Index

In [ ]:
embedding_dim = passage_embeddings.shape[1]

cpu_index = faiss.IndexFlatIP(embedding_dim)
gpu_res = faiss.StandardGpuResources()
rag_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
rag_index.add(passage_embeddings)

print(f"FAISS GPU index size: {rag_index.ntotal} vectors, dim={embedding_dim}")

FAISS GPU index size: 25577 vectors, dim=1024


### Saving Bundle

In [ ]:
out_dir = RAG_BUILD_CONFIG["output_dir"]

cpu_index_to_save = faiss.index_gpu_to_cpu(rag_index)
faiss.write_index(cpu_index_to_save, os.path.join(out_dir, "rag_index.faiss"))
passages_df.to_json(os.path.join(out_dir, "rag_passages.jsonl"), orient="records", lines=True)

manifest = {
    "embedding_model_name": RAG_BUILD_CONFIG["embedding_model_name"],
    "query_instruction": RAG_BUILD_CONFIG["query_instruction"],
    "reranker_model_name": RAG_BUILD_CONFIG["reranker_model_name"],
    "embedding_dim": embedding_dim,
    "num_passages": len(passages_df),
    "num_source_articles": len(title_to_text),
    "num_source_queries": len(unique_queries),
    "used_test_csv": "test_df" in dir(),
    "chunk_words": RAG_BUILD_CONFIG["chunk_words"],
    "chunk_overlap_words": RAG_BUILD_CONFIG["chunk_overlap_words"],
    "index_type": "IndexFlatIP",
}
with open(os.path.join(out_dir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

{
  "embedding_model_name": "BAAI/bge-large-en-v1.5",
  "query_instruction": "Represent this sentence for searching relevant passages: ",
  "reranker_model_name": "BAAI/bge-reranker-base",
  "embedding_dim": 1024,
  "num_passages": 25577,
  "num_source_articles": 486,
  "num_source_queries": 424,
  "used_test_csv": true,
  "chunk_words": 150,
  "chunk_overlap_words": 30,
  "index_type": "IndexFlatIP"
}


### Creating Zip

In [ ]:
bundle_zip_path = RAG_BUILD_CONFIG["bundle_zip_name"] + ".zip"
shutil.make_archive(RAG_BUILD_CONFIG["bundle_zip_name"], "zip", out_dir)
print("Created", bundle_zip_path)

Created rag_wiki_bundle.zip
